# 03 — Gold multitask QLoRA training

We fine-tune one Qwen adapter for region classification and transcription using the gold training data prepared in notebook 01. The baseline from notebook 02 provides a reference for later comparison; validation predictions are never used as training labels.

The initial run uses 64 regions to check supervision, numerical stability, memory use, and checkpoint saving. This smoke adapter is not the final trained model. Notebook 04 checks adapter loading and evaluates its predictions before full training.

Existing checkpoints are reused when the data and training settings match. Completed training is skipped, interrupted training resumes from its checkpoint, and a new run starts when no saved run exists.

In [ ]:
import importlib.metadata as metadata
import json
import sys
from pathlib import Path

import torch

CWD = Path.cwd().resolve()
CANDIDATES = [candidate for parent in (CWD, *CWD.parents) for candidate in (parent, parent / "src")]
PACKAGE = next(
    (
        candidate
        for candidate in CANDIDATES
        if (candidate / "rukopys" / "train.py").is_file() and (candidate / "work").is_dir()
    ),
    None,
)

if PACKAGE is None:
    raise FileNotFoundError(
        "Open the notebook from the project directory or one of its subdirectories. "
        "Place the rukopys package inside the existing src directory, next to work."
    )

if str(PACKAGE) not in sys.path:
    sys.path.insert(0, str(PACKAGE))

from rukopys.config import CLASS_PROMPT, PROMPTS
from rukopys.models import load_qwen
from rukopys.train import supervised_batch, train, training_examples
from rukopys.utils import digest, read_json

ROOT = (PACKAGE / "work").resolve()

for name in ("project.json", "data/train.json", "data/validation.json"):
    if not (ROOT / name).is_file():
        raise FileNotFoundError(f"Missing prepared artifact: work/{name}")

print("Python:", sys.version.split()[0])
print("Source directory:", PACKAGE.name)
print("Experiment directory:", ROOT.relative_to(PACKAGE))

Python: 3.12.5
Source directory: src
Experiment directory: work


## Check the environment

We reuse the working Python environment without automatically reinstalling packages. A fresh kernel keeps a previously loaded baseline model from occupying GPU memory.

The environment versions are displayed first. CUDA is required only when training actually starts or resumes; a completed checkpoint can be inspected without loading the model.

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

for name in (
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
):
    print(f"{name}: {metadata.version(name)}")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total VRAM (GiB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    )

PyTorch: 2.7.1+cu126
CUDA available: True
transformers: 4.57.6
peft: 0.21.2
accelerate: 1.15.0
bitsandbytes: 0.49.0
huggingface_hub: 0.36.2
GPU: NVIDIA GeForce GTX 1080 Ti
Total VRAM (GiB): 11.0


## Configure training

We freeze the base model weights, including the vision encoder, and train LoRA parameters in the language model's attention projections. The base model uses 4-bit NF4 quantization, with loss computed only on assistant answer tokens.

Each selected region contributes one classification example. Scorable regions also contribute a transcription example: legible Ukrainian content excluding images and graphs. The 64-region subset therefore contains more than 64 training examples.

The initial settings remain `RUN_NAME = "multitask_smoke"` and `MAX_REGIONS = 64`. For full training, restart the kernel, choose `RUN_NAME = "gold_multitask"`, and set `MAX_REGIONS = None`. Validation remains separate.

With `RESUME = None`, an existing run automatically uses the checkpoint recorded in `latest.json`. Completion is determined from the saved epoch and position, rather than the `COMPLETE` marker alone: that marker indicates a fully written checkpoint, not necessarily finished training.

A data or configuration mismatch stops execution. An existing run without a complete checkpoint also stops instead of silently restarting it.

In [ ]:
RUN_NAME = "multitask_smoke"
MAX_REGIONS = 64
EPOCHS = 1
LR = 1e-4
RANK = 8
ACCUMULATION = 4
MAX_PIXELS = 512 * 512
SAVE_EVERY = 10
RESUME = None  # Automatically use latest.json when a saved run exists

RUN_DIR = ROOT / "runs" / RUN_NAME
CONFIG_PATH = RUN_DIR / "config.json"
LATEST_PATH = RUN_DIR / "latest.json"

train_rows = read_json(ROOT / "data" / "train.json")
validation_rows = read_json(ROOT / "data" / "validation.json")
project = read_json(ROOT / "project.json")

if {row["page_id"] for row in train_rows} & {row["page_id"] for row in validation_rows}:
    raise ValueError("Train and validation pages overlap.")

selected_rows = (
    sorted(train_rows, key=lambda row: digest([42, row["id"]]))[:MAX_REGIONS]
    if MAX_REGIONS
    else train_rows
)
examples = training_examples(selected_rows)

if not examples:
    raise ValueError("No training examples are available.")

TRAINING_COMPLETE = False
SAVED_CONFIG = None
CHECKPOINT = None

if RESUME is None and LATEST_PATH.is_file():
    RESUME = read_json(LATEST_PATH)["checkpoint"]

if RESUME is not None:
    checkpoint = Path(RESUME).resolve()

    if checkpoint.parent != RUN_DIR.resolve():
        raise ValueError("The checkpoint must belong to the selected run directory.")

    for name in (
        "COMPLETE",
        "run_config.json",
        "adapter_config.json",
        "adapter_model.safetensors",
        "training_state.pt",
    ):
        if not (checkpoint / name).is_file():
            raise FileNotFoundError(f"Incomplete checkpoint: missing {name}.")

    SAVED_CONFIG = read_json(checkpoint / "run_config.json")
    saved_protocol = SAVED_CONFIG["protocol"]
    EXPECTED_CONFIG = {
        "protocol": saved_protocol,
        "model_revision": saved_protocol["model_revision"],
        "examples_sha256": digest(examples),
        "seed": project["seed"],
        "epochs": EPOCHS,
        "lr": LR,
        "rank": RANK,
        "alpha": RANK * 2,
        "dropout": 0.05,
        "accumulation": ACCUMULATION,
        "prompts": PROMPTS,
        "class_prompt": CLASS_PROMPT,
        "task_weighting": (
            "one classification example per region and "
            "one transcription example per scorable region"
        ),
    }

    if SAVED_CONFIG != EXPECTED_CONFIG:
        raise ValueError(
            "Saved training data or settings differ. "
            "Keep the existing run and choose a new RUN_NAME for a new experiment."
        )

    if CONFIG_PATH.is_file() and read_json(CONFIG_PATH) != SAVED_CONFIG:
        raise ValueError("Run and checkpoint configurations differ.")

    if (
        saved_protocol.get("model_id") != project["model_id"]
        or saved_protocol.get("model_revision") != project.get("model_revision")
        or saved_protocol.get("quantized") is not True
        or saved_protocol.get("max_pixels") != MAX_PIXELS
        or saved_protocol.get("adapter_sha256") is not None
    ):
        raise ValueError("The saved checkpoint uses a different base model protocol.")

    # Load only the local checkpoint state to inspect progress on the CPU.
    training_state = torch.load(
        checkpoint / "training_state.pt",
        map_location="cpu",
        weights_only=True,
    )
    saved_epoch = training_state["epoch"]
    saved_position = training_state["position"]

    if not (0 <= saved_epoch <= EPOCHS and 0 <= saved_position <= len(examples)):
        raise ValueError("The checkpoint contains invalid training progress.")

    TRAINING_COMPLETE = saved_epoch == EPOCHS or (
        saved_epoch == EPOCHS - 1 and saved_position == len(examples)
    )
    CHECKPOINT = str(checkpoint)
    RESUME = CHECKPOINT

    print("Saved optimizer updates:", training_state["updates"])
    print("Saved epoch:", saved_epoch)
    print("Saved position:", saved_position)
    del training_state

elif RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise RuntimeError(
        "The run directory exists but no checkpoint was selected. "
        "Restore latest.json, set RESUME to a complete checkpoint, "
        "or choose a new RUN_NAME."
    )

print("Available training regions:", len(train_rows))
print("Selected training regions:", len(selected_rows))
print("Classification examples:", sum(e["task"] == "classification" for e in examples))
print("Transcription examples:", sum(e["task"] == "transcription" for e in examples))
print("Total training examples:", len(examples))
print("Run directory:", RUN_DIR.relative_to(PACKAGE))
print("Training already complete:", TRAINING_COMPLETE)
print("Resume checkpoint:", Path(RESUME).name if RESUME else None)

Saved optimizer updates: 31
Saved epoch: 1
Saved position: 0
Available training regions: 20569
Selected training regions: 64
Classification examples: 64
Transcription examples: 59
Total training examples: 123
Run directory: work\runs\multitask_smoke
Training already complete: True
Resume checkpoint: checkpoint_000031


In [4]:
if TRAINING_COMPLETE:
    protocol = SAVED_CONFIG["protocol"]
    print("Training is already complete; model loading is skipped.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA-enabled kernel to start or resume training.")

    print("CUDA computation test:", (torch.ones(3, device="cuda") * 2).cpu().tolist())
    model, processor, protocol = load_qwen(
        ROOT,
        quantized=True,
        max_pixels=MAX_PIXELS,
    )

    if SAVED_CONFIG is not None and protocol != SAVED_CONFIG["protocol"]:
        raise ValueError(
            "The current model protocol differs from the checkpoint. "
            "Restore the original environment and settings before resuming."
        )

    print("Allocated VRAM (GiB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

print(json.dumps(protocol, ensure_ascii=False, indent=2))

Training is already complete; model loading is skipped.
{
  "model_id": "Qwen/Qwen3-VL-4B-Instruct",
  "model_revision": "ebb281ec70b05090aa6165b016eac8ec08e71b17",
  "quantized": true,
  "dtype": "torch.bfloat16",
  "max_pixels": 262144,
  "adapter_sha256": null,
  "packages": {
    "torch": "2.7.1+cu126",
    "transformers": "4.57.6",
    "peft": "0.21.2",
    "bitsandbytes": "0.49.0",
    "pillow": "11.1.0"
  }
}


## Audit supervision

Before new or resumed training, we inspect one example from each task. The supervised tokens should represent the assistant answer rather than the user prompt.

This audit checks label construction. It does not test a backward pass or guarantee that training fits in GPU memory. For completed runs, the audit is skipped because no model is loaded; previous outputs remain in the saved notebook until the cell is executed again.

In [5]:
if TRAINING_COMPLETE:
    print("Supervision audit skipped: the selected training run is complete.")
else:
    for task in ("classification", "transcription"):
        example = next((e for e in examples if e["task"] == task), None)

        if example is None:
            print(f"No {task} example in the selected subset.")
            continue

        batch = supervised_batch(processor, example, model.device)
        labels = batch["labels"][0]
        visible = labels[labels != -100]

        if visible.numel() == 0:
            raise RuntimeError(f"No supervised tokens for {task}.")

        print("Task:", task)
        print("Expected answer:", repr(example["answer"]))
        print(
            "Supervised tokens:",
            repr(processor.tokenizer.decode(visible, skip_special_tokens=False)),
        )

        del batch, labels, visible

Supervision audit skipped: the selected training run is complete.


## Training

We train only when the selected run is unfinished. An existing complete checkpoint supplies the adapter, processor, optimizer, scaler, random states, and training position for resuming.

The training cell should start at most once per kernel. After an interruption, restart the kernel and execute the notebook again with the same settings. Work after the most recent checkpoint must be repeated; training is not saved after every crop.

When all requested examples and epochs are already covered, the cell reuses the checkpoint path without calling the trainer or changing saved files. This confirms saved progress, not model quality.

In [6]:
if TRAINING_COMPLETE:
    print("Reusing completed training; no optimizer updates are needed.")
else:
    if globals().get("TRAINING_STARTED", False):
        raise RuntimeError(
            "Training has already started in this kernel. "
            "Restart the kernel before starting or resuming another training call."
        )

    TRAINING_STARTED = True
    model, CHECKPOINT = train(
        ROOT,
        model,
        processor,
        protocol,
        run_name=RUN_NAME,
        epochs=EPOCHS,
        lr=LR,
        rank=RANK,
        accumulation=ACCUMULATION,
        resume=RESUME,
        max_regions=MAX_REGIONS,
        save_every=SAVE_EVERY,
    )
    TRAINING_COMPLETE = True

if not (Path(CHECKPOINT) / "COMPLETE").is_file():
    raise RuntimeError("No complete checkpoint is available.")

print("Saved checkpoint:", Path(CHECKPOINT).relative_to(PACKAGE))

if LATEST_PATH.is_file():
    print("Latest recorded checkpoint:", Path(read_json(LATEST_PATH)["checkpoint"]).name)

Reusing completed training; no optimizer updates are needed.
Saved checkpoint: work\runs\multitask_smoke\checkpoint_000031
Latest recorded checkpoint: checkpoint_000031


## Next steps

We use notebook 04 with the same `RUN_NAME` to verify adapter loading and evaluate its predictions. A successful smoke test confirms that training runs; it does not establish improved recognition quality.

For full training, restart the kernel, set `RUN_NAME = "gold_multitask"` and `MAX_REGIONS = None`, and run this notebook again. The full-trained model is compared with the baseline using the same validation data and evaluation settings.

To resume an interrupted run, keep the entire run directory and the original settings. The notebook automatically selects the checkpoint recorded in `runs/<RUN_NAME>/latest.json`; `RESUME` can also select a checkpoint explicitly. Data preparation and baseline evaluation do not need to be repeated.

A complete checkpoint whose epoch and position cover the requested training is reused without loading Qwen. This reuse path checks saved data identities and settings, while actual training also verifies the selected crop hashes and current runtime protocol.